# Customer Churn Prediction

This notebook builds a machine learning model to predict customer churn for a bank.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')

## Step 1: Load Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('dataset/Churn_Modelling.csv')

# Display dataset shape
print(f"Dataset Shape: {df.shape}")

In [ ]:
# Display first 5 rows
df.head()

In [ ]:
# Display column names
print("Column Names:")
print(df.columns.tolist())

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())

In [ ]:
# Dataset info
df.info()

## Step 2: Data Cleaning & Preprocessing

In [ ]:
# Drop unnecessary columns
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

print(f"Shape after dropping columns: {df.shape}")

In [ ]:
# Encode categorical variables
le = LabelEncoder()
df['Geography'] = le.fit_transform(df['Geography'])
df['Gender'] = le.fit_transform(df['Gender'])

# Display preview
df.head()

In [ ]:
# Check datatypes after encoding
df.dtypes

## Step 3: Exploratory Data Analysis

In [ ]:
# Churn distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='Exited', data=df)
plt.title('Churn Distribution')
plt.xlabel('Exited (0=Stay, 1=Churn)')
plt.ylabel('Count')
plt.savefig('outputs/churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Gender vs Churn
plt.figure(figsize=(6, 4))
sns.countplot(x='Gender', hue='Exited', data=df)
plt.title('Gender vs Churn')
plt.xlabel('Gender (0=Female, 1=Male)')
plt.ylabel('Count')
plt.legend(['Stay', 'Churn'])
plt.savefig('outputs/gender_vs_churn.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Age distribution
plt.figure(figsize=(8, 4))
sns.histplot(df['Age'], bins=30, kde=True)
plt.title('Age Distribution')
plt.xlabel('Age')
plt.ylabel('Count')
plt.savefig('outputs/age_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.savefig('outputs/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Balance vs Churn
plt.figure(figsize=(8, 4))
sns.boxplot(x='Exited', y='Balance', data=df)
plt.title('Balance vs Churn')
plt.xlabel('Exited (0=Stay, 1=Churn)')
plt.ylabel('Balance')
plt.savefig('outputs/balance_vs_churn.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4: Feature Engineering

In [ ]:
# Separate features and target
X = df.drop('Exited', axis=1)
y = df['Exited']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling completed")

## Step 5: Model Training

In [ ]:
# Logistic Regression
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)
lr_accuracy = accuracy_score(y_test, lr_pred)
print(f"Logistic Regression Accuracy: {lr_accuracy:.4f}")

In [ ]:
# Random Forest Classifier
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)
rf_accuracy = accuracy_score(y_test, rf_pred)
print(f"Random Forest Accuracy: {rf_accuracy:.4f}")

In [ ]:
# Gradient Boosting Classifier
gb_model = GradientBoostingClassifier(random_state=42)
gb_model.fit(X_train_scaled, y_train)
gb_pred = gb_model.predict(X_test_scaled)
gb_accuracy = accuracy_score(y_test, gb_pred)
print(f"Gradient Boosting Accuracy: {gb_accuracy:.4f}")

In [ ]:
# Model comparison table
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'Gradient Boosting'],
    'Accuracy': [lr_accuracy, rf_accuracy, gb_accuracy]
})
results

## Step 6: Model Evaluation (Best Model)

In [ ]:
# Identify best model
best_model_name = results.loc[results['Accuracy'].idxmax(), 'Model']
print(f"Best Model: {best_model_name}")

In [ ]:
# Use best model for evaluation (Random Forest)
best_pred = rf_pred

# Evaluation metrics
accuracy = accuracy_score(y_test, best_pred)
precision = precision_score(y_test, best_pred)
recall = recall_score(y_test, best_pred)
f1 = f1_score(y_test, best_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

In [ ]:
# Classification report
print("Classification Report:")
print(classification_report(y_test, best_pred))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, best_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.savefig('outputs/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 7: Model Comparison Visualization

In [ ]:
# Model accuracy comparison
plt.figure(figsize=(8, 4))
sns.barplot(x='Model', y='Accuracy', data=results)
plt.title('Model Accuracy Comparison')
plt.xlabel('Model')
plt.ylabel('Accuracy')
plt.ylim(0.7, 1.0)
for i, v in enumerate(results['Accuracy']):
    plt.text(i, v + 0.01, f'{v:.4f}', ha='center')
plt.savefig('outputs/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 8: Customer Churn Prediction System

In [ ]:
def predict_churn(credit_score, geography, gender, age, tenure, balance, 
                 num_products, has_cr_card, is_active_member, estimated_salary):
    """
    Predict customer churn status.
    
    Parameters:
    - credit_score: Customer's credit score
    - geography: Customer's location (0=France, 1=Germany, 2=Spain)
    - gender: Customer's gender (0=Female, 1=Male)
    - age: Customer's age
    - tenure: Number of years with bank
    - balance: Account balance
    - num_products: Number of products
    - has_cr_card: Has credit card (0=No, 1=Yes)
    - is_active_member: Is active member (0=No, 1=Yes)
    - estimated_salary: Estimated salary
    
    Returns: Prediction message
    """
    # Create feature array
    features = np.array([[credit_score, geography, gender, age, tenure, balance, 
                          num_products, has_cr_card, is_active_member, estimated_salary]])
    
    # Scale features
    features_scaled = scaler.transform(features)
    
    # Predict
    prediction = rf_model.predict(features_scaled)[0]
    
    if prediction == 1:
        return "Customer likely to churn"
    else:
        return "Customer likely to stay"

In [ ]:
# Example prediction 1
result1 = predict_churn(credit_score=600, geography=0, gender=1, age=40, tenure=5, 
                        balance=50000, num_products=2, has_cr_card=1, is_active_member=1, 
                        estimated_salary=50000)
print(f"Prediction 1: {result1}")

In [ ]:
# Example prediction 2
result2 = predict_churn(credit_score=400, geography=1, gender=0, age=50, tenure=2, 
                        balance=0, num_products=1, has_cr_card=0, is_active_member=0, 
                        estimated_salary=30000)
print(f"Prediction 2: {result2}")

## Summary

This notebook successfully:
- Loaded and preprocessed the customer churn dataset
- Performed exploratory data analysis with visualizations
- Trained and compared three machine learning models
- Evaluated the best model using multiple metrics
- Created a prediction function for new customers

The Random Forest model achieved the highest accuracy and was used for the final prediction system.